## Install library dependencies

***USE PYTHON 3.10**  
Run in terminal:  
pip install pandas scikit-learn everywhereml tensorflow

### Import libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

## Setup

In [2]:
# 1. Load dataset
df = pd.read_csv('../Dataset/DATA TRAINING.csv')

# 2. Choose features and encode target labels as integers (required by everywhereml export)
X = df[['Start Time (s)', 'End Time (s)', 'RMSSD (ms)', 'SDNN (ms)', 'BPM', 'Num R-peaks']] # features
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Label Stres'].astype(str).str.strip()) # encoded target variable

# 3. Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Show label mapping (int -> original label)
print('Label mapping:', dict(enumerate(label_encoder.classes_)))

# Configure model parameters
rf_model = RandomForestClassifier(
    n_estimators=50,      # <--- PARAMETER: How many trees in your forest! (More trees = stronger, but slower)
    max_depth=None,        # <--- PARAMETER: How deep the roots go! (None means they grow until they are perfect)
    min_samples_split=2,   # <--- PARAMETER: Minimum samples needed to split a node
    min_samples_leaf=1,    # <--- PARAMETER: Minimum samples needed to be a leaf
    random_state=42,       # <--- PARAMETER: Random seed so you get the same results every time
    n_jobs=-1              # <--- PARAMETER: How many N jobs to run in parallel. Higher values use more resources.
)

Label mapping: {0: 'stres rendah', 1: 'stres sedang', 2: 'stres tinggi'}


## Train model

In [3]:
# 4. Train the model
rf_model.fit(X_train, y_train)

# 5. Make predictions
y_pred = rf_model.predict(X_test)

# 6. Print results
print("Model Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Details:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

Model Performance:
Accuracy: 100.00%

Details:
              precision    recall  f1-score   support

stres rendah       1.00      1.00      1.00        24
stres sedang       1.00      1.00      1.00        54
stres tinggi       1.00      1.00      1.00        19

    accuracy                           1.00        97
   macro avg       1.00      1.00      1.00        97
weighted avg       1.00      1.00      1.00        97



## Export model
After running this cell:  
1) Copy the generated code  
2) Make a new file to store the code (this is the exported model)    
3) Paste it  

In [4]:
"""
Export Random Forest model to C++ header code for Arduino.
This mirrors the trained rf_model hyperparameters and exports the model as a C++ class.
"""
from everywhereml.sklearn.ensemble import RandomForestClassifier as EmlRandomForestClassifier

# Build an Everywhereml-compatible RF model using the same parameters as rf_model
eml_rf_model = EmlRandomForestClassifier(**rf_model.get_params())
eml_rf_model.fit(X_train, y_train)

# Generate C++ header code
c_header = eml_rf_model.to_cpp(class_name='RandomForestModel')

# Save generated header to file
output_file = '../Model/rf_model.h'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(c_header)

print(f"Model exported to {output_file}")

Model exported to ../Model/rf_model.h
